In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from numpy import cos,pi,sqrt
from astropy.table import Table
from astropy.table import unique
import os
import ctadata
from astropy.io import ascii

In [2]:
gheff='gheff0.9'


In [3]:
run_catalog=ascii.read('LST_source_catalog.ecsv')
run_catalog.columns

good_runs=np.load('good_runs_Zd_80.0.npy')
1874 in good_runs

True

In [4]:
cat=ascii.read('run_catalog_dark_nights2800.dat')
cat.columns
r=cat['Run ID']
rr=cat['RA_PNT']
dd=cat['DEC_PNT']
cc=cat['FILENAME']
tt=cat['Exposure[s]']
ss=cat['SRC_NAME']
aa=cat['ALT_PNT']
zz=cat['AZ_PNT']
tt=cat['TSTART[MJD]']
nn=cat['NSB_TUNING']
pp=cat['POINT/FULL_ENCLOSURE']

runs=[]
ra_pnts=[]
dec_pnts=[]
flist=[]
texpos=[]
src_names=[]
alt_pnts=[]
az_pnts=[]
ra_objs=[]
dec_objs=[]
tstarts=[] 
nsb_tunings=[]
point_fulls=[]

for i in range(len(r)):
    runs.append(r[i])
    ra_pnts.append(rr[i])
    dec_pnts.append(dd[i])
    texpos.append(tt[i])
    alt_pnts.append(aa[i])
    az_pnts.append(zz[i])
    tstarts.append(tt[i])
    src_names.append(ss[i])
    nsb_tunings.append(nn[i])
    point_fulls.append(pp[i])
    flist.append(cc[i])
    
len(cat),len(flist)

(2800, 2800)

In [6]:
last_run=runs[-1]
last_run

11074

In [7]:
dates_DL3=ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/')

In [ ]:
dates=[]
counter=len(flist)
print(counter)
for i in range(len(run_catalog)):
    run=run_catalog[i]['Run ID']
    if(run>last_run):
        src=run_catalog[i]['Source name']
        date=run_catalog[i]['Date directory'].replace('-','')
        if(date in dates_DL3)and(run in good_runs): 
            vers=ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date)
            for ver in vers:
                if(ver[0]=='v'):
                    tailcuts=ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver)
                    for tailcut in (tailcuts):
                        nsbs=ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut)
                        for nsb in nsbs:
                            versions=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb))
                            for version in versions:
                                tags=ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std')
                                for tag in tags:
                                    src_dependences=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag))
                                    for src_dep in src_dependences:
                                        point_or_full=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep))
                                        for p_f in point_or_full:
                                            wobbles=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f))
                                            for wob in wobbles:
                                                gheffs=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob))        
                                                for gh in gheffs:
                                                    if(gheff in gh):
                                                        irfs=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh))
                                                        for irf in irfs:
                                                            files=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf))
                                                            if(run<10000):
                                                                fname='dl3_LST-1.Run0'+str(run)+'.fits'
                                                            else:
                                                                fname='dl3_LST-1.Run'+str(run)+'.fits'
                                                            if(fname in files):
                                                                runs.append(run)
                                                                flist.append('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf+'/'+fname)
                                                                f=('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf+'/'+fname)
                                                                counter+=1
                                                                ctadata.fetch_and_save_file_or_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf+'/'+fname)
                                                                hdul=fits.open(fname)
                                                                header=hdul['EVENTS'].header
                                                                texpos.append(header['LIVETIME'])
                                                                src_names.append(src)
                                                                alt_pnts.append(header['ALT_PNT'])
                                                                az_pnts.append(header['AZ_PNT'])
                                                                ra_objs.append(header['RA_OBJ'])
                                                                dec_objs.append(header['DEC_OBJ'])
                                                                mjdref=header['MJDREFI']
                                                                nsb_tunings.append(nsb[-4:])
                                                                tstarts.append(mjdref+header['TSTART']/86400)
                                                                dates.append(date)
                                                                point_fulls.append(p_f)
                                                                print(counter,date,fname,src_names[-1],texpos[-1],p_f)
                                                                ra_pnts.append(header['RA_PNT'])
                                                                dec_pnts.append(header['DEC_PNT'])
    
                                                                if(counter%100==0):
                                                                    cat = Table()
                                                                    cat['Run ID'] = runs
                                                                    cat['RA_PNT'] = ra_pnts
                                                                    cat['DEC_PNT']=dec_pnts
                                                                    cat['Exposure[s]']=texpos
                                                                    cat['ALT_PNT']=alt_pnts
                                                                    cat['AZ_PNT']=az_pnts
                                                                    cat['TSTART[MJD]']=tstarts
                                                                    cat['SRC_NAME']=src_names
                                                                    cat['NSB_TUNING']=nsb_tunings
                                                                    cat['POINT/FULL_ENCLOSURE']=point_fulls
                                                                    cat['FILENAME']=flist
                                                                    ascii.write(cat, 'run_catalog_dark_nights'+str(counter)+'.dat', overwrite=True)
        
                                                                os.remove(fname)
    #else:
    #    print(date+' not in dCache')


2800
2801 20221126 dl3_LST-1.Run11075.fits G106.3+2.7 1101.0764395181245 full-enclosure
2802 20221126 dl3_LST-1.Run11075.fits G106.3+2.7 1101.0764395181245 full-enclosure
2803 20221126 dl3_LST-1.Run11075.fits G106.3+2.7 1101.0764395181245 point
2804 20221126 dl3_LST-1.Run11076.fits G106.3+2.7 1109.7700982488166 full-enclosure
2805 20221126 dl3_LST-1.Run11076.fits G106.3+2.7 1109.7700982488166 point
2806 20221126 dl3_LST-1.Run11077.fits G106.3+2.7 1108.8177644238085 full-enclosure
2807 20221126 dl3_LST-1.Run11077.fits G106.3+2.7 1108.8177644238085 full-enclosure
2808 20221126 dl3_LST-1.Run11077.fits G106.3+2.7 1108.8177644238085 point
2809 20221126 dl3_LST-1.Run11078.fits G106.3+2.7 1105.1010537728348 full-enclosure
2810 20221126 dl3_LST-1.Run11078.fits G106.3+2.7 1105.1010537728348 point
2811 20221126 dl3_LST-1.Run11079.fits G106.3+2.7 1095.3316480726282 full-enclosure
2812 20221126 dl3_LST-1.Run11079.fits G106.3+2.7 1095.3316480726282 full-enclosure
2813 20221126 dl3_LST-1.Run11079.fi

In [ ]:
cat = Table()
cat['Run ID'] = runs 
cat['RA_PNT'] = ra_pnts
cat['DEC_PNT']=dec_pnts
cat['Exposure[s]']=texpos
cat['ALT_PNT']=alt_pnts
cat['AZ_PNT']=az_pnts
cat['TSTART[MJD]']=tstarts
cat['SRC_NAME']=src_names
cat['NSB_TUNING']=nsb_tunings
cat['POINT/FULL_ENCLOSURE']=point_fulls
cat['FILENAME']=flist
ascii.write(cat, 'run_catalog_dark_nights.dat', overwrite=True)

In [ ]:
ascii.read('run_catalog_dark_nights.dat')